In [ ]:


%pip install ipympl==0.9.3


  Using cached ipympl-0.9.3-py2.py3-none-any.whl.metadata (1.3 kB)
  Using cached ipython_genutils-0.2.0-py2.py3-none-any.whl.metadata (755 bytes)
Using cached ipympl-0.9.3-py2.py3-none-any.whl (511 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.8/831.8 kB 23.6 MB/s  0:00:00
Using cached ipython_genutils-0.2.0-py2.py3-none-any.whl (26 kB)
  Attempting uninstall: ipython
    Found existing installation: ipython 9.11.0
    Uninstalling ipython-9.11.0:
      Successfully uninstalled ipython-9.11.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipympl]2m2/3 [ipympl]]
Note: you may need to restart the kernel to use updated packages.


# Interferometry
## SAR Image
Single image is 2D 
- Azimuth coordinate (from absolute time)  corresponds to a horizontal coordinate on Earth
- Range coordinate (from return time) corresponds to a mixed horizontal/vertical coordinate

<div style="display: flex; gap: 20px;">
    <img src="figures/amplitude.png" style="width: 50%;">
    <img src="figures/phase.png" style="width: 50%;">
</div>

## SAR SLC observations

SLC: Single-Look Complex data
- Single-look: no averageing, finest spatial resolution
- Complex: both real and imaginary (In-Phase and quadrature phase)

A given pixel in a SAR image is a comples number where both aplitude and phase have a physical interpretable origin.


$$
y_1 = \underbrace{|y_1|}_{\text{Amplitude}} \exp(j\underbrace{\psi_1}_{\text{Phase}})
$$




```{figure} figures/sinusoidal.png
---
height: 300px
name: signal

---

**Amplitude** corresponds to the amount of radiated energy reflected back to the sensor and so shows up as the brightness of the pixel. The **phase** component of the pixel corresponds to the travel time of the received radar pulse and thus a function of distance of the target to the antenna.



The Phase components are represented as:

$$
\underline{\psi} = \psi_{\text{geom}}+ \psi_{\text{scat}}+\psi_{\text{atmo}}+\underline{\psi}_{\text{noise}}
$$

This phase combination of several phase components results in what is essentially a random number uniformly distributed on the interval $[-\pi, \pi)$.

The most important components here are the geometric phase $\psi_{\text{geom}}$ and the scattering phase $\psi_{\text{scat}}$

The observed value of a resolution cell in a SAR image is the sum of all the backscattered energy of the objects within that cell, called elementary scatterers. Certain objects will reflect a radar pulse back to the sensor more effectively due to their material properties, shape, size, and orientation. Different configurations of elementary scatterers will result in different observed behaviours. These scatterers can be catagorized into groups according to their spatial and temporal characteristics.

In [ ]:

#hide this cell
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import rayleigh
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

def make_scatterers(N, A, seed):
    rng = np.random.default_rng(seed)
    amp = rng.uniform(0, A, N)
    pha = rng.uniform(-np.pi, np.pi, N)
    I = amp * np.cos(pha)
    Q = amp * np.sin(pha)
    colors = rng.uniform(0.1, 0.85, (N, 3))
    return I, Q, colors

def draw(N, A, seed):
    I, Q, colors = make_scatterers(N, A, seed)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))
    fig.subplots_adjust(wspace=0.3)

    # --- Left: individual arrows from origin ---
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)
    ax1.axhline(0, color='gray', lw=0.5)
    ax1.axvline(0, color='gray', lw=0.5)
    for i in range(N):
        ax1.annotate("", xy=(I[i], Q[i]), xytext=(0, 0),
                     arrowprops=dict(arrowstyle="-|>", color=colors[i],
                                     lw=2, mutation_scale=12))
    lim = A * 1.25
    ax1.set_xlim(-lim, lim)
    ax1.set_ylim(-lim, lim)
    ax1.set_title(f'Individual scatterers (N={N})', fontsize=11)
    ax1.set_xlabel('I')
    ax1.set_ylabel('Q')

    # --- Right: tip-to-tail + red resultant ---
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)
    ax2.axhline(0, color='gray', lw=0.5)
    ax2.axvline(0, color='gray', lw=0.5)

    si, sq = 0.0, 0.0
    for i in range(N):
        ax2.annotate("", xy=(si + I[i], sq + Q[i]), xytext=(si, sq),
                     arrowprops=dict(arrowstyle="-|>", color=colors[i],
                                     lw=2, mutation_scale=12))
        si += I[i]
        sq += Q[i]

    ax2.annotate("", xy=(si, sq), xytext=(0, 0),
                 arrowprops=dict(arrowstyle="-|>", color='red',
                                 lw=3, mutation_scale=18))

    all_x = np.cumsum(np.concatenate([[0], I]))
    all_y = np.cumsum(np.concatenate([[0], Q]))
    margin = max(np.max(np.abs(all_x)), np.max(np.abs(all_y)), 0.5) * 1.2
    ax2.set_xlim(-margin, margin)
    ax2.set_ylim(-margin, margin)
    resultant = np.sqrt(si**2 + sq**2)
    ax2.set_title(f'Vector sum  |Amplitude resultant| = {resultant:.2f}', fontsize=11)
    ax2.set_xlabel('I')
    ax2.set_ylabel('Q')

    plt.show()


# ── Widgets ───────────────────────────────────────────────────────────────────

state = {'seed': 0}

slider_N = widgets.IntSlider(value=10, min=1, max=20, step=1,
                             description='N:', continuous_update=False)
slider_A = widgets.FloatSlider(value=3.0, min=1.0, max=5.0, step=0.1,
                               description='A:', continuous_update=False,
                               readout_format='.1f')
btn_new  = widgets.Button(description='New sample', button_style='')

out = widgets.Output()

def redraw(*args):
    out.clear_output(wait=True)
    with out:
        draw(slider_N.value, slider_A.value, state['seed'])

def new_sample(b):
    state['seed'] += 1
    redraw()

slider_N.observe(redraw, names='value')
slider_A.observe(redraw, names='value')
btn_new.on_click(new_sample)

controls = widgets.HBox([slider_N, slider_A, btn_new])
display(controls, out)
redraw()

Output()

### Point Scatterers

Resolution cells containing an object with a high radar cross-section (RCS) will be dominated by the reflections from that object, as shown in Fig. 2.2. As a result, the behaviour of the entire resolution cell can be localized to the point where that object is located, with all other elementary scatterers acting as clutter. Such a scatterer configuration is called a point scatterer (PS)
This type of analysis is called point scatterer interferometry (PSI)

```{figure} figures/ps_scatterer.png
---
height: 300px
name: psi

---
```


### Distributed Scatterers

```{figure} figures/ds_scatterer.png
---
height: 300px
name: dsi

---
```

```{figure} figures/phase_diff.png
---
height: 300px
name: phase difference

---
```

## Interferometric phase observation

To get geometric information a combination of multiple coherent SAR acquistions of the same scene need to be used.  The term Coferent refers to the phases of the SAR acquisitions that are preserved, so that a coherent combination of two SAR image pixels can be the interference os the two samples wave returns of a given target. The coherent combination of two SAR phase images is called an interferogram, and the phase difference between two SAR pixels is called the interferometric phase. 

In order for the interferometric phase to contain any useful geometric information, there needs to be a difference between the two SAR images that are combined, either spatial or temporal. This difference is called the interferometric baseline, and the choice of this baseline will affect the type of information that is contained within the interferometric phase.

Using the interferometric configuration sketched in fig we derive the physical and the geometrical relationships between the two phase observations to obtain topographic height and surface deformation estimates.


$$
y_1 = |y_1|exp(j\psi_1)
$$
$$
y_2 = |y_2|exp(j\psi_2)
$$
$$
\mu=y_1y_2^*= |y_1||y_2|\exp{(j \psi_1-\psi_2)}
$$



$$
\frac{2R_1}{\lambda}= \frac{\psi_1}{2\pi}
$$

$$
\psi_1 = -\frac{4\pi}{\lambda}R_1
$$
$$
\psi_2 = -\frac{4\pi}{\lambda}R_2
$$

Interferometric phase (for given pixel $p$):

$$
\phi_p = \psi_{1p}-\psi_{2p}= - \frac{4\pi(R_1-R_2)}{\lambda}= - \frac{4\pi\Delta R}{\lambda}
$$

$$
\underline{\phi_p} = \phi_{\text{geom}}+ \phi_{\text{scat}}+\phi_{\text{atmo}}+\underline{\phi}_{\text{noise}}
$$

The individual components are the the phase contributions due to the differences:
- $\phi_{\text{geom}}$ in geometry
- $\phi_{\text{scat}}$ in scattering
- $\phi_{\text{atmo}}$ in athmosphere 
- $\underline{\phi}_{\text{noise}}$ is the sum of the noise in both pixels $p$






### Coherence

In [ ]:


N = 10   # fixed
A = 1.0  # fixed

def make_acquisitions(gamma, seed):
    rng     = np.random.default_rng(seed)
    amps    = A*np.ones(N)#rng.uniform(0, A, N)
    phases1 = rng.uniform(-np.pi, np.pi, N)
    colors  = rng.uniform(0.1, 0.85, (N, 3))
    noise   = rng.uniform(-np.pi, np.pi, N)
    phases2 = phases1 + (1 - gamma) * noise
    vecs1   = np.stack([amps * np.cos(phases1), amps * np.sin(phases1)], axis=1)
    vecs2   = np.stack([amps * np.cos(phases2), amps * np.sin(phases2)], axis=1)
    return vecs1, vecs2, colors

def draw_chain(ax, vecs, colors, resultant_color, extent):
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    ax.axhline(0, color='gray', lw=0.5)
    ax.axvline(0, color='gray', lw=0.5)
    si, sq = 0.0, 0.0
    for v, col in zip(vecs, colors):
        ax.annotate("", xy=(si + v[0], sq + v[1]), xytext=(si, sq),
                    arrowprops=dict(arrowstyle="-|>", color=col,
                                   lw=1.8, mutation_scale=12))
        si += v[0]; sq += v[1]
    ax.annotate("", xy=(si, sq), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color=resultant_color,
                               lw=3.5, mutation_scale=18))
    ax.set_xlim(-extent, extent)
    ax.set_ylim(-extent, extent)
    return si, sq

def draw(gamma, seed):
    vecs1, vecs2, colors = make_acquisitions(gamma, seed)

    s1I, s1Q = vecs1[:, 0].sum(), vecs1[:, 1].sum()
    s2I, s2Q = vecs2[:, 0].sum(), vecs2[:, 1].sum()
    dI,  dQ  = s2I - s1I, s2Q - s1Q

    # Shared axis extent for plots 1 & 2
    all_coords = np.abs(np.concatenate([
        np.cumsum(np.concatenate([[0], vecs1[:, 0]])),
        np.cumsum(np.concatenate([[0], vecs1[:, 1]])),
        np.cumsum(np.concatenate([[0], vecs2[:, 0]])),
        np.cumsum(np.concatenate([[0], vecs2[:, 1]])),
    ]))
    extent12 = max(all_coords.max(), 0.5) * 1.25

    r1       = np.sqrt(s1I**2 + s1Q**2)
    r2       = np.sqrt(s2I**2 + s2Q**2)
    ph1      = np.arctan2(s1Q, s1I)
    ph2      = np.arctan2(s2Q, s2I)
    diff_mag = np.sqrt(dI**2 + dQ**2)
    dph      = ph2 - ph1
    # if dph >  180: dph -= 360
    # if dph < -180: dph += 360
    extent3  = max(r1, r2, diff_mag, 0.5) * 1.35

    fig, axes = plt.subplots(1, 3, figsize=(16, 6))
    fig.suptitle(f'Coherence  γ = {gamma:.2f}', fontsize=13, y=1.01)
    fig.subplots_adjust(wspace=0.4)

    # Plot 1 — Acquisition 1
    draw_chain(axes[0], vecs1, colors, '#1a7fc1', extent12)
    axes[0].set_title(f'Acquisition 1\n|Y₁| = {r1:.2f}   φ₁ = {ph1:.1f}rad', fontsize=11)
    axes[0].set_xlabel('I'); axes[0].set_ylabel('Q')

    # Plot 2 — Acquisition 2
    draw_chain(axes[1], vecs2, colors, '#c17a1a', extent12)
    axes[1].set_title(f'Acquisition 2\n|Y₂| = {r2:.2f}   φ₂ = {ph2:.1f}rad', fontsize=11)
    axes[1].set_xlabel('I'); axes[1].set_ylabel('Q')

    # Plot 3 — Difference
    axes[2].set_aspect('equal')
    axes[2].grid(True, alpha=0.2)
    axes[2].axhline(0, color='gray', lw=0.5)
    axes[2].axvline(0, color='gray', lw=0.5)

    # Faded resultants for reference
    axes[2].annotate("", xy=(s1I, s1Q), xytext=(0, 0),
                     arrowprops=dict(arrowstyle="-|>", color='#1a7fc1',
                                    lw=2.5, mutation_scale=14, alpha=0.3))
    axes[2].annotate("", xy=(s2I, s2Q), xytext=(0, 0),
                     arrowprops=dict(arrowstyle="-|>", color='#c17a1a',
                                    lw=2.5, mutation_scale=14, alpha=0.3))
    # Difference vector tip-to-tip
    axes[2].annotate("", xy=(s2I, s2Q), xytext=(s1I, s1Q),
                     arrowprops=dict(arrowstyle="-|>", color='#a32d2d',
                                    lw=3.5, mutation_scale=18))
    axes[2].set_xlim(-extent3, extent3)
    axes[2].set_ylim(-extent3, extent3)
    axes[2].set_title(f'Difference  (Y₂ − Y₁)\n|ΔY| = {diff_mag:.2f}   Δφ = {dph:.1f}rad', fontsize=11)
    axes[2].set_xlabel('I'); axes[2].set_ylabel('Q')

    plt.show()


# ── Widgets ───────────────────────────────────────────────────────────────────

state = {'seed': 0}

slider_gamma = widgets.FloatSlider(
    value=0.8, min=0.0, max=1.0, step=0.01,
    description='γ (coherence):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px'),
    continuous_update=False,
    readout_format='.2f'
)
btn_new = widgets.Button(description='New sample', button_style='')

out = widgets.Output()

def redraw(*args):
    out.clear_output(wait=True)
    with out:
        draw(slider_gamma.value, state['seed'])

def new_sample(b):
    state['seed'] += 1
    redraw()

slider_gamma.observe(redraw, names='value')
btn_new.on_click(new_sample)

display(widgets.HBox([slider_gamma, btn_new]), out)
redraw()

Output()

We are interested in the geometric phase which consists of 3 sub -components:

$$
\phi_{\text{geom}}= \phi_{\text{disp}}+\phi_{\text{ref}}+\underline{\phi}_{\text{topo}}
$$

- $\phi_{\text{disp}}$ is the displacement phase
- $\phi_{\text{ref}}$ is the reference phase
- $\phi_{\text{topo}}$ is the topographic phase

```{figure} figures/interferometric_configuration.png
---
height: 300px
name: interf_config

---

```



### Topographic Phase

$$
\phi_p =  - \frac{4\pi\Delta R}{\lambda}
$$

$$
\partial \phi_p= -\frac{4 \pi}{\lambda}\partial \Delta R
$$

$$
\Delta R = B\sin(\theta-\alpha)
$$
$$
\partial \Delta R = B\cos(\theta^{\circ}-\alpha)\partial \theta
$$
initial value of $\theta^{\circ}$ is obtained for the reference surface
$$
\partial \phi_p = -\frac{4 \pi}{\lambda} B\cos(\theta^{\circ}-\alpha)\partial \theta
$$



$$
H_p =  -\frac{\lambda R_1 \sin\theta^{\circ} }{4 \pi B_{\perp}} \partial\phi_p
$$

With $B_{\perp}= B\cos(\theta^{\circ}-\alpha)$

### Reference Phase

$$
\phi_{ref}= \frac{4\pi}{\lambda}B\sin(\theta - \alpha) 
$$

